
# 17 — Prompt Engineering for Document Summarization

**Learning reference:**
[BrightPool — Prompt Engineering Course](https://github.com/BrightPool/udemy-prompt-engineering-course)

**Adapted from:**
- Progressive summarization
- Summarizing large documents
- Direct (stuff) and map-reduce summarization

## Goal

Compare two prompt-based summarization strategies
using real Gemini model responses.

1. Direct summarization: summarize the full document.
2. Map-reduce summarization: summarize document
   sections separately, then combine their summaries.

We will evaluate whether both methods preserve
important facts from the original document.

The experiment uses a fictional report with
predefined reference facts.


The experiment uses a small controlled document
with known reference facts, making the two strategies
easy to compare. Evaluation is performed locally.


Four model requests are planned:

- One direct summary.
- Two section summaries.
- One final combined summary.




## 1. Summarization Strategies

### Direct summarization (Stuff)

The entire document is placed in one prompt.

Advantages:
- Simple to implement.
- One model request.
- The model sees the full source at once.

Limitations:
- The document must fit in the context window.
- A short summary may omit important details.

### Map-reduce summarization

The document is divided into smaller chunks.

**Map:** Summarize each chunk independently.

**Reduce:** Combine the chunk summaries.

Advantages:
- Can process documents larger than a single
  model request's practical input budget.
- Individual sections are easier to process.

Limitations:
- Multiple model requests are required.
- Information lost during chunk summarization
  cannot reliably be recovered at the reduce stage.
- The final summary may omit context needed
  to understand relationships between sections.

### Progressive / Refine summarization

A third approach processes chunks sequentially,
updating an existing summary with each new chunk.

It is related to map-reduce but is not identical.

We will explain Refine conceptually and implement
direct and map-reduce approaches for comparison.

This notebook tests summarization prompts,
not model training or fine-tuning.


In [1]:

sections = [
    {
        "title": "1. Revenue",
        "text": (
            "HarborCart is a fictional online retailer. "
            "In the first quarter of 2026, its revenue "
            "increased by 12% compared with the previous "
            "quarter. Management attributed the growth "
            "primarily to repeat customers and a new "
            "subscription offering."
        )
    },
    {
        "title": "2. Customer Support",
        "text": (
            "The support team introduced a new ticket "
            "triage process. The median first-response "
            "time decreased to 48 hours from 72 hours. "
            "Customer complaints about slow responses "
            "remained a concern despite the improvement."
        )
    },
    {
        "title": "3. Product Returns",
        "text": (
            "HarborCart changed its standard return "
            "window from 30 days to 14 days. The change "
            "reduced processing workload, but customers "
            "reported that the shorter period made "
            "returns less convenient."
        )
    },
    {
        "title": "4. Delivery",
        "text": (
            "The company introduced an express delivery "
            "service costing €8 per order. This service "
            "is available only in selected cities. "
            "Standard delivery remains available "
            "outside those areas."
        )
    },
    {
        "title": "5. Mobile Application",
        "text": (
            "A redesign of the checkout process reduced "
            "mobile checkout abandonment by 20%. "
            "However, some users continued to report "
            "login failures after the update."
        )
    },
    {
        "title": "6. Next Quarter Priorities",
        "text": (
            "The leadership team plans to focus on "
            "support response times, mobile login "
            "reliability, and customer dissatisfaction "
            "with the shorter return window. "
            "No expansion date has been announced "
            "for express delivery."
        )
    }
]

document = "\n\n".join(
    f"{section['title']}\n{section['text']}"
    for section in sections
)

print(document)
print("\nSections:", len(sections))
print("Characters:", len(document))

assert len(sections) == 6


1. Revenue
HarborCart is a fictional online retailer. In the first quarter of 2026, its revenue increased by 12% compared with the previous quarter. Management attributed the growth primarily to repeat customers and a new subscription offering.

2. Customer Support
The support team introduced a new ticket triage process. The median first-response time decreased to 48 hours from 72 hours. Customer complaints about slow responses remained a concern despite the improvement.

3. Product Returns
HarborCart changed its standard return window from 30 days to 14 days. The change reduced processing workload, but customers reported that the shorter period made returns less convenient.

4. Delivery
The company introduced an express delivery service costing €8 per order. This service is available only in selected cities. Standard delivery remains available outside those areas.

5. Mobile Application
A redesign of the checkout process reduced mobile checkout abandonment by 20%. However, some users 


## 2. Define the Evaluation Criteria

Before calling Gemini, we identify important
facts from the source report.

The summarization prompts will prioritize:

- Numerical accuracy.
- Preservation of important policy changes.
- Clear distinction between improvements
  and unresolved problems.
- No unsupported claims.

Six reference facts will be checked:

1. Revenue increased by 12%.
2. Median support response time fell to 48 hours.
3. The return window changed to 14 days.
4. Express delivery costs €8.
5. Checkout abandonment fell by 20%.
6. Express delivery has no announced expansion date.

We will search model outputs for these facts
using simple text patterns.

These checks measure selected factual coverage,
not complete summary quality.

We will also inspect the outputs manually for
hallucinations, missing context and misleading
wording.


In [2]:

import re

REFERENCE_FACTS = {
    "Revenue growth": r"\b12\s*%",
    "Support response time": r"\b48\s*hours?\b",
    "Return window": r"\b14\s*days?\b",
    "Express delivery price": r"(?:€\s*8\b|\b8\s*euros?\b)",
    "Checkout improvement": r"\b20\s*%",
    "Expansion date not announced": (
        r"(?:no|not|without).{0,65}"
        r"(?:expansion date|date.{0,25}expansion)"
        r"|expansion.{0,50}(?:no|not|unannounced)"
    )
}

assert len(REFERENCE_FACTS) == 6

print("Reference facts prepared:")
for name in REFERENCE_FACTS:
    print("-", name)


Reference facts prepared:
- Revenue growth
- Support response time
- Return window
- Express delivery price
- Checkout improvement
- Expansion date not announced


In [3]:

%pip install -q "langchain-core>=1,<2" "langchain-openai>=1,<2"

from getpass import getpass

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = "gemini-3.5-flash-lite"

gemini_key = getpass("Enter your Gemini API key: ")

gemini = ChatOpenAI(
    model=MODEL,
    api_key=gemini_key,
    base_url=(
        "https://generativelanguage.googleapis.com/"
        "v1beta/openai/"
    ),
    temperature=0,
    max_retries=0,
    timeout=120,
    use_responses_api=False
)

parser = StrOutputParser()

print("Gemini configured:", MODEL)
print("No model requests made.")


Enter your Gemini API key: ··········
Gemini configured: gemini-3.5-flash-lite
No model requests made.



## 3. Direct Summarization

The first approach sends the entire report to Gemini.

The prompt specifies:

- Role: business-report summarization assistant.
- Objective: produce a concise executive summary.
- Grounding: use only the supplied document.
- Accuracy: preserve numerical values and units.
- Balance: mention improvements and unresolved issues.
- Length: target 120–160 words.

We will not tell Gemini exactly which six facts
appear in our local evaluation.

This keeps the evaluation criteria separate
from the prompt and lets us observe which facts
the model naturally preserves.



In [4]:

direct_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are an executive-report summarization assistant.

Create a concise, factual summary of the supplied
business report.

Rules:
- Use only information from the document.
- Preserve important numerical values and units.
- Include major improvements, policy changes,
  remaining problems and future priorities.
- Do not invent explanations, dates or results.
- Distinguish completed changes from planned work.
- Write approximately 120 to 160 words.
- Use clear, professional English.
"""
    ),
    (
        "human",
        "Summarize this report:\n\n{document}"
    )
])

direct_chain = (
    direct_prompt
    | gemini
    | parser
)

if "direct_summary" not in globals():
    direct_summary = direct_chain.invoke({
        "document": document
    })
    print("Direct summary generated.")
else:
    print("Using saved direct summary.")

print("\nDIRECT SUMMARY\n")
print(direct_summary)
print("\nWords:", len(direct_summary.split()))

assert direct_summary.strip()


Direct summary generated.

DIRECT SUMMARY

In the first quarter of 2026, HarborCart achieved a 12% revenue increase over the previous quarter, driven primarily by repeat customers and a new subscription offering. 

Completed operational and policy changes include introducing a €8 express delivery service in selected cities (with standard delivery remaining elsewhere) and shortening the standard return window from 30 days to 14 days, which successfully reduced processing workload. Additionally, the support team implemented a new ticket triage process, cutting the median first-response time from 72 hours down to 48 hours. A mobile application checkout redesign decreased mobile checkout abandonment by 20%.

Remaining problems involve customer dissatisfaction regarding the less convenient shorter return window, ongoing complaints about slow support responses, and persistent user-reported login failures following the mobile update. Future priorities for leadership include improving support 


## 4. Prepare Map-Reduce Summarization

We will divide the six sections into two chunks:

**Chunk 1**
- Revenue
- Customer support
- Returns

**Chunk 2**
- Delivery
- Mobile application
- Next-quarter priorities

The two chunks are processed independently.

Each chunk receives a prompt that requests
a factual intermediate summary.

The final combine prompt sees those intermediate
summaries, not the original report.

This is important: if the intermediate summaries
omit information, the final model cannot reliably
recover it from the source.

The source is small, so chunking is for an
experimental comparison, not because the document
exceeds Gemini's context window.


In [5]:

def render_sections(items):
    return "\n\n".join(
        f"{item['title']}\n{item['text']}"
        for item in items
    )


chunks = [
    render_sections(sections[:3]),
    render_sections(sections[3:])
]

assert len(chunks) == 2

print("Chunk 1 characters:", len(chunks[0]))
print("Chunk 2 characters:", len(chunks[1]))

map_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You summarize one section of a larger business report.

Write a concise intermediate summary.

Rules:
- Use only the supplied text.
- Preserve exact figures, units and policy changes.
- Include both improvements and unresolved problems.
- Preserve statements about unknown or unannounced plans.
- Do not add facts not contained in the text.
- Target 70 to 100 words.
"""
    ),
    (
        "human",
        "Summarize this report excerpt:\n\n{chunk}"
    )
])

map_chain = map_prompt | gemini | parser

print("\nMap-stage prompt constructed.")
print("Model requests: 0 in this cell.")


Chunk 1 characters: 683
Chunk 2 characters: 603

Map-stage prompt constructed.
Model requests: 0 in this cell.


In [6]:

if "chunk_1_summary" not in globals():
    chunk_1_summary = map_chain.invoke({
        "chunk": chunks[0]
    })
    print("Chunk 1 summarized.")
else:
    print("Using saved chunk 1 summary.")

print("\nCHUNK 1 SUMMARY\n")
print(chunk_1_summary)

assert chunk_1_summary.strip()


Chunk 1 summarized.

CHUNK 1 SUMMARY

In Q1 2026, HarborCart’s revenue grew 12% quarter-over-quarter, driven by repeat customers and a new subscription offering. Customer support implemented a ticket triage process, reducing the median first-response time from 72 to 48 hours; however, complaints about slow responses remained a concern. HarborCart also shortened its standard return window from 30 days to 14 days. While this policy change reduced processing workload, customers reported that the shorter period decreased return convenience.


In [7]:

if "chunk_2_summary" not in globals():
    chunk_2_summary = map_chain.invoke({
        "chunk": chunks[1]
    })
    print("Chunk 2 summarized.")
else:
    print("Using saved chunk 2 summary.")

print("\nCHUNK 2 SUMMARY\n")
print(chunk_2_summary)

assert chunk_2_summary.strip()


Chunk 2 summarized.

CHUNK 2 SUMMARY

The company launched an €8 express delivery service in selected cities, while standard delivery continues elsewhere; no expansion date has been announced. A mobile application checkout redesign reduced mobile checkout abandonment by 20%, though some users still experience login failures. Next quarter, leadership will prioritize support response times, mobile login reliability, and customer dissatisfaction regarding the shorter return window.



## 5. Reduce: Combine Intermediate Summaries

The final step merges the two chunk summaries.

The reduce prompt asks Gemini to:

- Produce one coherent executive summary.
- Preserve numerical facts available in the inputs.
- Avoid duplication.
- Include major improvements and open issues.
- Avoid inventing facts that were omitted earlier.

The model does not receive the original document.

This allows us to examine information loss
between the map and reduce stages.

The final summary should be similar in length
to the direct summary.


In [8]:

combine_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are consolidating summaries of consecutive
sections of one business report.

Create a single executive summary.

Rules:
- Use only the provided intermediate summaries.
- Preserve the numerical facts and units available.
- Preserve important policy changes and unresolved issues.
- Avoid repeating the same information.
- Do not invent details to fill gaps.
- Distinguish achievements from future plans.
- Write approximately 120 to 160 words.
"""
    ),
    (
        "human",
        "Part 1 summary:\n{part_1}\n\n"
        "Part 2 summary:\n{part_2}"
    )
])

combine_chain = combine_prompt | gemini | parser

if "combined_summary" not in globals():
    combined_summary = combine_chain.invoke({
        "part_1": chunk_1_summary,
        "part_2": chunk_2_summary
    })
    print("Combined summary generated.")
else:
    print("Using saved combined summary.")

print("\nMAP-REDUCE SUMMARY\n")
print(combined_summary)
print("\nWords:", len(combined_summary.split()))

assert combined_summary.strip()


Combined summary generated.

MAP-REDUCE SUMMARY

In Q1 2026, HarborCart achieved 12% quarter-over-quarter revenue growth, driven by repeat customers and a new subscription offering. Achievements include implementing a ticket triage process that reduced median first-response time from 72 to 48 hours, shortening the standard return window from 30 days to 14 days to reduce processing workload, launching an €8 express delivery service in selected cities alongside continued standard delivery, and redesigning the mobile application checkout, which reduced mobile checkout abandonment by 20%. 

However, unresolved issues remain: customers complained about slow support responses and decreased return convenience due to the shorter window, and some users still experience mobile login failures. No expansion date has been announced for express delivery. Next quarter, leadership will prioritize improving support response times, mobile login reliability, and customer dissatisfaction regarding the ret

In [9]:

def evaluate_summary(text):
    return {
        name: bool(
            re.search(
                pattern,
                text,
                flags=re.IGNORECASE | re.DOTALL
            )
        )
        for name, pattern in REFERENCE_FACTS.items()
    }


direct_checks = evaluate_summary(direct_summary)
combined_checks = evaluate_summary(combined_summary)

# Inspect the intermediate summaries separately
# to detect possible information loss.
map_stage_text = (
    chunk_1_summary + "\n" + chunk_2_summary
)

map_checks = evaluate_summary(map_stage_text)

rows = []

for fact in REFERENCE_FACTS:
    rows.append({
        "Reference fact": fact,
        "Direct": direct_checks[fact],
        "Map stage": map_checks[fact],
        "Final map-reduce": combined_checks[fact]
    })

results_df = pd.DataFrame(rows) if "pd" in globals() else None

if results_df is None:
    import pandas as pd
    results_df = pd.DataFrame(rows)

print(results_df.to_string(index=False))

direct_score = sum(direct_checks.values())
map_score = sum(map_checks.values())
combined_score = sum(combined_checks.values())

total = len(REFERENCE_FACTS)

print("\nFACT COVERAGE")
print(f"Direct summary: {direct_score}/{total}")
print(f"Map stage: {map_score}/{total}")
print(f"Final map-reduce: {combined_score}/{total}")

lost_in_reduce = [
    fact
    for fact in REFERENCE_FACTS
    if map_checks[fact] and not combined_checks[fact]
]

print("\nFacts matched at map stage but not in final:")
print(lost_in_reduce or "None detected")

print("\nSummary lengths:")
print("Direct:", len(direct_summary.split()), "words")
print("Map-reduce:", len(combined_summary.split()), "words")

if direct_score > combined_score:
    print("\nDirect summary preserved more checked facts.")
elif combined_score > direct_score:
    print("\nMap-reduce preserved more checked facts.")
else:
    print("\nBoth preserved the same number of checked facts.")

print("\nIMPORTANT:")
print("Keyword matches are only screening checks.")
print("Manually check the context and direction of each fact.")
print("Also inspect both summaries for unsupported claims.")

print("\nAdditional Gemini requests: 0")


              Reference fact  Direct  Map stage  Final map-reduce
              Revenue growth    True       True              True
       Support response time    True       True              True
               Return window    True       True              True
      Express delivery price    True       True              True
        Checkout improvement    True       True              True
Expansion date not announced    True       True              True

FACT COVERAGE
Direct summary: 6/6
Map stage: 6/6
Final map-reduce: 6/6

Facts matched at map stage but not in final:
None detected

Summary lengths:
Direct: 151 words
Map-reduce: 131 words

Both preserved the same number of checked facts.

IMPORTANT:
Keyword matches are only screening checks.
Manually check the context and direction of each fact.
Also inspect both summaries for unsupported claims.

Additional Gemini requests: 0



## Findings and Limitations

### What This Notebook Demonstrates

- Designing prompts for factual document summarization.
- Direct summarization of a complete source document.
- Splitting documents into sections.
- Map-reduce summarization using real Gemini calls.
- Combining model-generated intermediate summaries.
- Checking whether factual information survives
  multiple summarization stages.
- Comparing strategies against the same predefined
  reference facts.

### Prompt Engineering Techniques

**Grounding**

The prompts restrict Gemini to the supplied source.

**Numerical accuracy**

Instructions emphasize preserving figures and units.

**Content prioritization**

The prompts ask for major changes, improvements,
outstanding problems and future priorities.

**Length constraints**

Both final summaries target approximately
120–160 words.

**Multi-stage prompting**

The map-reduce strategy passes model-generated
summaries into another model prompt.

**Evaluation**

Predetermined reference facts are checked in
the generated outputs.

### Relationship to the Udemy Examples

This notebook combines the course's progressive
summarization and large-document summarization
ideas in a smaller Gemini-based experiment.

The original examples also discuss:

- Token counting.
- Context-window limits.
- Adaptive chunk sizes.
- Recursive summarization.
- Retrying with smaller inputs when an oversized
  request is rejected.

These concepts are important for production
systems, but are not fully implemented here.

This experiment uses fixed, section-based chunks
to focus on prompt design and information
preservation.

### Observed Results

Gemini generated a direct summary, two intermediate
section summaries, and a final map-reduce summary.

| Measurement | Direct | Map stage | Final map-reduce |
|---|---:|---:|---:|
| Reference-fact pattern matches | 6/6 | 6/6 | 6/6 |
| Final summary length | 151 words | — | 131 words |

All six checks matched:

- Revenue increased by 12%.
- Median first-response time decreased to 48 hours.
- The return window changed to 14 days.
- Express delivery costs €8.
- Mobile checkout abandonment decreased by 20%.
- No express delivery expansion date was announced.

No checked facts were lost between the intermediate
summaries and the final map-reduce summary.

Both final summaries distinguished completed
improvements from ongoing problems and planned
priorities.

A manual comparison against the fictional source
found no apparent invented factual claims.
However, the automated checks alone do not
establish complete factual accuracy.

**Interpretation**

Both approaches retained all six selected facts.

Map-reduce provided no measurable fact-coverage
advantage on this short source document.
The direct summary was slightly longer.

This result does not establish which strategy
would perform better on larger documents.

### Limitations

- The report is fictional and relatively short.
- Only one source document is tested.
- Each approach is executed once.
- The approaches differ in prompt wording and
  document-processing structure, so results
  cannot isolate chunking as the sole cause.
- Both final summaries use approximate
  word-count instructions.
- Fact coverage relies on simple text patterns.
- Pattern matches do not prove factual correctness.
- Pattern mismatches can miss valid paraphrases.
- Hallucinations require additional manual review.
- The example does not demonstrate handling
  truly oversized documents.
- Findings cannot be generalized to longer
  documents or other language models.

### Conclusion

This notebook demonstrates how prompt design and
multi-stage processing influence document
summarization.

Direct summarization gives the model access
to the entire source in a single stage.

Map-reduce summarization processes sections
individually, then combines their summaries.

Both approaches preserved all six selected facts
in this experiment.

The main lesson is that summarization quality
depends on what the prompt asks the model to
preserve, what context the model receives,
and how information passes between stages.
